In [1]:
import numpy as np
import sys
import os

# Add parent directory to sys.path
parent_path = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if parent_path not in sys.path:
    sys.path.append(parent_path)

# Import data processing functions from src/data_processing.py
from src.data_processing import load_csv, one_hot_encoding, ordinal_encoding,\
    strip_quotes, convert_numeric

Load raw data from `data/raw/BankChurner.csv`

In [2]:
DATA_PATH = "../data/raw/BankChurners.csv"
OUT_DIR = "../data/processed"
data, header = load_csv(DATA_PATH)

Remove double quotes ( " )

In [3]:
v_strip = np.vectorize(strip_quotes)
header = list(v_strip(header))
data = v_strip(data)

Convert numeric values to appropriate types

In [4]:
v_convert_numeric = np.vectorize(convert_numeric, otypes=[object])
data = v_convert_numeric(data)

Handle 'Unknown' values and encode categorical features

- `Gender`: Gender 
    - Has two values `F` and `M`.

    ➡ Label Encoding as `0` and `1`


- `Attrition_Flag`: Customer status flag (staying/leaving).
    - Has 2 values `"Existing Customer"` and `"Attrited Customer"`.

    ➡ Label Encoding as `0` and `1`

- `Education_Level`: Education level, ordered from low to high
    - `Uneducated`: 1
    - `High School`: 2
    - `College`: 3
    - `Graduate`: 4
    - `Post-Graduate`: 5
    - `Doctorate`: 6

    ➡ Ordinal encoding.

- `Income_Category`: Income classification, ranges from low to high
    - `Less than $40K`: 1
    - `$40K - $60K`: 2
    - `$60K - $80K`: 3
    - `$80K - $120K`: 4
    - `$120K +`: 5

    ➡ Ordinal encoding.

- `Marital_Status`: Marital status, no clear ordering

    ➡ One-hot encoding.

- `Card_Category`: Card type, order not necessarily important (lower card type doesn't necessarily mean higher churn rate).
    - `Blue`: 0
    - `Silver`: 1
    - `Gold`: 2
    - `Platinum`: 3

    ➡ Ordinal encoding.

In [5]:
cat_col_names = ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category', 'Attrition_Flag']
cat_col_idx = [header.index(c) for c in cat_col_names]

# Attrition_Flag
Attrition_idx = header.index('Attrition_Flag')
# Attrited - 1, Existing - 0
feat_attrition = np.where(data[:, Attrition_idx] == 'Attrited Customer', 1, 0).reshape(-1, 1)

# Gender
Gender_idx = header.index('Gender')
# M - 1, F - 0
feat_gender = np.where(data[:, Gender_idx] == 'M', 1, 0).reshape(-1, 1)

# Education level 
Edu_idx = header.index('Education_Level')
edu_col = data[:, Edu_idx]
# Find mode value to replace 'Unknown' values
vals, counts = np.unique(edu_col[edu_col != 'Unknown'], return_counts=True)
edu_mode = vals[np.argmax(counts)]
edu_col[edu_col == 'Unknown'] = edu_mode
# Education level (Ordinal Encoding)
# Dictionary storing numeric values for encoding (ordered by education level from low to high)
Edu_dict = {'Uneducated': 1, 'High School': 2, 'College': 3, 'Graduate': 4, 'Post-Graduate': 5, 'Doctorate': 6}
feat_edu = ordinal_encoding(Edu_dict, edu_col).reshape(-1, 1)

# Income Category
inc_idx = header.index('Income_Category')
inc_col = data[:, inc_idx]
# Find mode value to replace 'Unknown' values
vals, counts = np.unique(inc_col[inc_col != 'Unknown'], return_counts=True)
inc_mode = vals[np.argmax(counts)]
inc_col[inc_col == 'Unknown'] = inc_mode
# Income Category (Ordinal Encoding)
# Dictionary storing numeric values for encoding (ordered by income amount from low to high)
inc_dict = {'Less than $40K': 1, '$40K - $60K': 2, '$60K - $80K': 3, '$80K - $120K': 4, '$120K +': 5}
feat_inc = ordinal_encoding(inc_dict, inc_col).reshape(-1, 1)


# Marital Status (one hot encoding)
marital_idx = header.index('Marital_Status')
feat_marital, marital_uni = one_hot_encoding(data[:, marital_idx])

# Card Category (one hot encoding)
card_idx = header.index('Card_Category')
# Dictionary storing numeric values for encoding (ordered by card rank from low to high)
card_dict = {'Blue': 0, 'Silver': 1, 'Gold': 2, 'Platinum': 3}
feat_card = ordinal_encoding(card_dict, data[:, card_idx]).reshape(-1, 1)

data_cat_encoded = np.hstack((feat_gender, feat_edu, feat_inc, feat_marital, feat_card))
data_cat_encoded[:5]

array([[1, 2, 3, 0, 1, 0, 0, 0],
       [0, 4, 1, 0, 0, 1, 0, 0],
       [1, 4, 4, 0, 1, 0, 0, 0],
       [0, 2, 1, 0, 0, 0, 1, 0],
       [1, 1, 3, 0, 1, 0, 0, 0]])

Handle outliers in numeric columns using IQR (Interquartile Range) method by capping to allowable range

In [6]:
num_col_names = [
    'Customer_Age', 'Dependent_count', 'Months_on_book', 
    'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 
    'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 
    'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 
    'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio'
]
num_col_idx = [header.index(c) for c in num_col_names] # list

# Handle outliers in numeric columns
data_num = data[:, num_col_idx]
Q1 = np.percentile(data_num, 25, axis=0)
Q3 = np.percentile(data_num, 75, axis=0)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
data_num_clipped = np.clip(data_num, lower_bound, upper_bound)

Feature Engineering

- `Avg_Trans_Value` - Average transaction value per transaction:
    - Reflects customer spending capability.

    - Customers with few transactions but high value usually: 

    - Have good income,

    - Use card for large purposes,

    - Tend to stay more.

    Conversely:

    If customer has low total spending and small transactions → high likelihood of leaving.

- `Activity_Ratio` - Transaction frequency over time:

    - This is the average transaction frequency per month, playing a key role in churn:

    - Customers with high activity ratio → use card regularly → less likely to leave

    - Customers with low activity ratio → decreasing transaction frequency → early sign of churn

    - Decreasing transaction frequency is one of the strong signals of churn.

In [7]:
# Feature Engineering
engineered_headers = ['Avg_Trans_Value', 'Activity_Ratio']
num_header_list = list(num_col_names)
idx_Total_Trans_Amt = num_header_list.index('Total_Trans_Amt')
idx_Total_Trans_Ct = num_header_list.index('Total_Trans_Ct')
idx_Months_on_book = num_header_list.index('Months_on_book')
safe_ct = data_num_clipped[:, idx_Total_Trans_Ct] + 1e-6
safe_months = data_num_clipped[:, idx_Months_on_book] + 1e-6

# Average transaction value
feat_avg_trans_val = (data_num_clipped[:, idx_Total_Trans_Amt] / safe_ct).reshape(-1, 1)
# Activity frequency
feat_activity_ratio = (data_num_clipped[:, idx_Total_Trans_Ct] / safe_months).reshape(-1, 1)

data_num_engineered = np.hstack((data_num_clipped, feat_avg_trans_val, feat_activity_ratio))
data_num_engineered

array([[45, 3, 39, ..., 0.061, 27.23809458956918, 1.0769230493096655],
       [49, 5, 44, ..., 0.105, 39.12121093572088, 0.7499999829545458],
       [51, 3, 36, ..., 0, 94.34999528250023, 0.5555555401234572],
       ...,
       [44, 1, 36, ..., 0, 143.6541642724306, 1.6666666203703717],
       [30, 2, 36, ..., 0, 135.40322362252866, 1.7222221743827175],
       [43, 2, 25, ..., 0.189, 141.2991780114889, 2.439999902400004]],
      dtype=object)

Apply Z-score normalization to encoded data to ensure absolute consistency and optimize performance for machine learning models later

In [8]:
X_final = np.hstack((data_cat_encoded, data_num_engineered))
X_final = X_final.astype(float)
mean = np.mean(X_final, axis=0)
std = np.std(X_final, axis=0)
safe_std = std + 1e-6
X_scaled = (X_final - mean) / safe_std
X_scaled

array([[ 1.05995352, -0.89367928,  0.59729936, ..., -0.77587942,
        -1.81037902, -0.95760241],
       [-0.94343381,  0.59338777, -0.88762776, ..., -0.61627342,
        -1.12064144, -1.33318114],
       [ 1.05995352,  0.59338777,  1.33976292, ..., -0.99715138,
         2.08503009, -1.55656457],
       ...,
       [-0.94343381, -0.89367928, -0.88762776, ..., -0.99715138,
         4.94681626, -0.28008784],
       [ 1.05995352,  0.59338777, -0.1451642 , ..., -0.99715138,
         4.46790285, -0.216264  ],
       [-0.94343381,  0.59338777, -0.88762776, ..., -0.31157105,
         4.81012463,  0.60833993]])

Save preprocessed data to `data/processed` directory

In [9]:
final_header_list = (
    cat_col_names[:1]  +     # "Gender"
    cat_col_names[1:2] +     # "Education_level"
    cat_col_names[3:4] +     # "Income_Category"
    list(marital_uni)  +     # Marital_status (One Hot Encoding)
    cat_col_names[4:5] +     # Card_Category
    num_col_names +          # 14 numeric columns
    engineered_headers +     # 2 new columns (feature engineering)
    cat_col_names[-1:]       # 'Attrition_Flag' target column
)
header_string = ','.join(final_header_list)

combined_data = np.hstack((X_scaled, feat_attrition))

out_dir = "../data/processed/BankChurners.csv"
out_path = os.path.dirname(out_dir)
os.makedirs(out_path, exist_ok=True)

np.savetxt(
    out_dir, 
    combined_data, 
    delimiter=',', 
    fmt='%.8f',
    header=header_string,  
    comments=''           
)